# Sites and Sweeps

Builds simple site collections and sweep inputs that later intervention notebooks can reuse.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.user_funcs.nn",
    "tl.user_funcs.normalize_hook_plan",
    "tl.user_funcs.normalize_input_args",
    "tl.user_funcs.os",
    "tl.user_funcs.pickle",
    "tl.user_funcs.random",
    "tl.user_funcs.record_kpi_in_graph",
    "tl.user_funcs.register_tensor_connection",
    "tl.user_funcs.reset_naming_counter",
    "tl.user_funcs.resolve_renamed_kwarg",
    "tl.user_funcs.safe_copy_args",
    "tl.user_funcs.safe_copy_kwargs",
    "tl.user_funcs.set_random_seed",
    "tl.user_funcs.show_backward_graph",
    "tl.user_funcs.show_bundle_graph",
    "tl.user_funcs.show_model_graph",
    "tl.user_funcs.summary",
    "tl.user_funcs.torch",
    "tl.user_funcs.tqdm",
    "tl.user_funcs.validate_backward_pass",
    "tl.user_funcs.validate_batch_of_models_and_inputs",
    "tl.user_funcs.validate_forward_pass",
    "tl.user_funcs.validate_saved_activations",
    "tl.user_funcs.validate_training_compatibility",
    "tl.user_funcs.visualization_to_render_kwargs",
    "tl.user_funcs.warn_deprecated_alias",
    "tl.user_funcs.warn_parallel",
    "tl.utils",
    "tl.utils.AutocastRestore",
    "tl.utils.DoctorCheck",
    "tl.utils.DoctorReport",
    "tl.utils.MAX_FLOATING_POINT_TOLERANCE",
    "tl.utils._ATTR_SKIP_SET",
    "tl.utils._AUTOCAST_DEVICES",
    "tl.utils._cuda_available",
    "tl.utils._get_func_call_stack",
    "tl.utils._is_cuda_available",
    "tl.utils._model_expects_single_arg",
    "tl.utils._safe_copy_arg",
    "tl.utils.assign_to_sequence_or_dict",
    "tl.utils.doctor",
    "tl.utils.ensure_iterable",
    "tl.utils.find_executable_save_set",
    "tl.utils.flop_count",
    "tl.utils.format_flops",
    "tl.utils.format_size",
    "tl.utils.get_attr_values_from_tensor_list",
    "tl.utils.get_tensor_memory_amount",
    "tl.utils.get_vars_of_type_from_obj",
    "tl.utils.human_readable_size",
    "tl.utils.identity",
    "tl.utils.in_notebook",
    "tl.utils.index_nested",
    "tl.utils.int_list_to_compact_str",
    "tl.utils.is_iterable",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")